1.Bronze processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/gayatrijoshi663@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","employees","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
# Define S3 path
bucket = "regis-healthcare"
prefix = f"source-row-data /{data_source}"
s3_path = f"s3://{bucket}/{prefix}/"

# Use dbutils to list files and find the latest
files = dbutils.fs.ls(s3_path)
latest_file = sorted(files, key=lambda x: x.modificationTime, reverse=True)[0]

base_path = latest_file.path
print("Latest file path:", base_path)


In [0]:
df = (
    spark.read.format("csv")
       .option("header",True)
       .option("inferSchema",True)
       .load(base_path)
       .withColumn("current_date",F.current_date())
       .withColumn("read_timestamp",F.current_timestamp())
       .select("*","_metadata.file_name","_metadata.file_size")
    )
print(df.count())
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
        .option("mergeSchema","true")\
     .option("overwriteSchema","true")\
         .option("delta.enableChangeDataFeed","true")\
             .mode("overwrite")\
                 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
# bronze write to s3
df.write.format("delta")\
    .option("mergeSchema","true")\
        .option("overwriteSchema","true")\
            .mode("overwrite")\
            .partitionBy("current_date")\
            .save(f"s3://regis-healthcare/bronze-row-data/{data_source}/")

2. Silver processing

In [0]:
df_bronze =spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze.limit(10))

In [0]:
# schema check
df_bronze.printSchema()

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()


In [0]:
df_silver.columns

In [0]:

df_silver = df_silver.withColumn(
    "employee_id",
    F.trim(F.col("employee_id"))
).withColumn(
    "first_name",
    F.trim(F.col("first_name"))
).withColumn(
    "last_name",
    F.trim(F.col("last_name"))
).withColumn(
    "role",
    F.trim(F.col("role"))
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "employment_type",
    F.trim(F.col("employment_type"))
).withColumn(
    "start_date",
    F.trim(F.col("start_date"))
).withColumn(
    "end_date",
    F.trim(F.col("end_date"))
).withColumn(
    "phone",
    F.trim(F.col("phone"))
).withColumn(
    "email",
    F.trim(F.col("email"))
).withColumn(
    "address",
    F.trim(F.col("address"))
).withColumn(
    "state",
    F.trim(F.col("state"))
).withColumn(
    "postcode",
    F.trim(F.col("postcode"))
).withColumn(
    "ahpra_number",
    F.trim(F.col("ahpra_number"))
).withColumn(
    "salary",
    F.trim(F.col("salary"))
).withColumn(
    "manager_id",
    F.trim(F.col("manager_id"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
).withColumn(
    "status",
    F.trim(F.col("status"))
)    

In [0]:
# null record count
from pyspark.sql.functions import col, count, when

null_count = df_silver.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_silver.columns
])
display(null_count)

Cleaning data in table 

In [0]:
# employee_id 
check = df_silver.filter(~col("employee_id").rlike("^EMP"))
display(check)

df_silver = df_silver.withColumn("employee_id",when(~col("employee_id").rlike("^EMP"),None).otherwise(col("employee_id")))
display(df_silver)

In [0]:
#  'first_name'

# from pyspark.sql.functions import col
# df=df_silver.filter(col("first_name").isNull())

# df_silver = df_silver.fillna({
#     "first_name":"unknown"
# })

from pyspark.sql import functions as F
from pyspark.sql.functions import col,when

df_invalid = df_silver.filter(col("first_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

df_silver = df_silver.withColumn("first_name",when(col("first_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"),"unknown").otherwise(F.initcap(F.trim(F.col("first_name")))))

df_silver = df_silver.withColumn("first_name",when(col("first_name").rlike("Null|N/a|Inf|Nan|\x00\x00|���|None"),"unknown").otherwise(F.initcap(F.trim(F.col("first_name")))))

df_invalid = df_silver.filter(col("first_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)
df_silver = df_silver.fillna({"first_name":"unknown"})
dup = df_silver.groupBy("first_name").count().filter(col("count")>1)
display(dup)

df_silver = df_silver.withColumn("first_name",when(col("first_name")== "\x00\x00","unknown").otherwise(F.initcap(F.trim(F.col("first_name")))))

display(df_silver)
print(df_silver.count())

In [0]:
#'last_name'
print(df_silver.count())
from pyspark.sql import functions as F
from pyspark.sql.functions import col,when

df_invalid = df_silver.filter(col("last_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

df_silver = df_silver.withColumn("last_name",when(col("last_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"),"unknown").otherwise(F.initcap(F.trim(F.col("last_name")))))

df_silver = df_silver.withColumn("last_name",when(col("last_name").rlike("Null|N/a|Inf|Nan|\x00\x00|���|None"),"unknown").otherwise(F.initcap(F.trim(F.col("last_name")))))

df_invalid = df_silver.filter(col("last_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)
df_silver = df_silver.fillna({"last_name":"unknown"})
dup = df_silver.groupBy("last_name").count().filter(col("count")>1)
display(dup)

df_silver = df_silver.withColumn("last_name",when(col("last_name")== "\x00\x00","unknown").otherwise(F.initcap(F.trim(F.col("last_name")))))

display(df_silver)
print(df_silver.count())

In [0]:
dup=df_silver.groupBy("role").count().filter(col("count")>1)
display(dup)



In [0]:
#  'facility_id'
# print(df_silver.count())
# from pyspark.sql import functions as F
# from pyspark.sql.functions import col,when
# df_silver = df_silver.withColumn("facility_id",when(~col("facility_id").rlike("^[0-9]+(\\.[0-9]+)?$"),None).otherwise(col("facility_id").cast("int")))

check = df_silver.filter(~col("facility_id").rlike("^FAC"))
display(check)
df_silver = df_silver.withColumn("facility_id",when(~col("facility_id").rlike("^FAC"),None).otherwise(col("facility_id")))
display(df_silver)

In [0]:
#  'employment_type
dup=df_silver.groupBy("role").count().filter(col("count")>1)
display(dup)

In [0]:
#  'start_date

from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = df_silver.withColumn(
    "start_date",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("start_date")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("start_date")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("start_date")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("start_date")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("start_date")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("start_date")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = df_silver.withColumn("start_date", F.to_date("start_date"))
display(df_silver)

In [0]:
#  'end_date
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F

# First, filter out "currently working" before date parsing
df_silver = df_silver.withColumn(
    "end_date_cleaned",
    when(F.lower(F.trim(col("end_date"))).isin(["currently working", "null", "n/a", "nan", "none", ""]), F.lit(None))
    .otherwise(F.trim(col("end_date")))
)

# Parse cleaned end_date to date format
df_silver = df_silver.withColumn(
    "end_date_parsed",
    F.coalesce(
        # Date-only formats
        F.try_to_date(col("end_date_cleaned"), F.lit("yyyy/MM/dd")),
        F.try_to_date(col("end_date_cleaned"), F.lit("dd/MM/yyyy")),
        F.try_to_date(col("end_date_cleaned"), F.lit("yyyy-MM-dd")),
        F.try_to_date(col("end_date_cleaned"), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.to_date(F.try_to_timestamp(col("end_date_cleaned"), F.lit("yyyy-MM-dd HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(col("end_date_cleaned"), F.lit("yyyy/MM/dd HH:mm:ss")))
    )
)

# Convert to string format, replacing nulls with "currently working"
df_silver = df_silver.withColumn(
    "end_date",
    when(col("end_date_parsed").isNull(), "currently working")
    .otherwise(F.date_format(col("end_date_parsed"), "yyyy-MM-dd"))
).drop("end_date_cleaned", "end_date_parsed")


display(df_silver)

In [0]:
#  'phone
# dup=df_silver.groupBy("phone").count().filter(col("count")>1)
# display(dup)

# df_invalid = df_silver.filter(~col("phone").rlike("[^0-9,{10}]$"))
# # display(df_invalid)

# df_silver = df_silver.withColumn("phone",when(col("phone").rlike("[^0-9,{10}]$"),"Unknown").otherwise(col("phone")))

# df_invalid = df_silver.filter(col("phone").rlike("[^0-9,{10}$]"))
# display(df_invalid)

# # df_silver = df_silver.fillna({"phone":"unknown"})


# df_silver = df_silver.withColumn("phone",when(col("phone")== "Invalid","Unknown").otherwise(col("phone")))


# df_silver = df_silver.withColumn("phone",when(col("phone").isNull(),"Unknown").otherwise(col("phone")))

# dup = df_silver.groupBy("phone").count().filter(col("count")>1)
# display(dup)

# # display(df_silver)
# print(df_silver.count())

In [0]:
from pyspark.sql.functions import col, when

# Define valid phone regex: exactly 10 digits
valid_phone_regex = "^[0-9]{10}$"

# Replace invalid or null phones with "Unknown"
df_silver = df_silver.withColumn(
    "phone",
    when(col("phone").rlike(valid_phone_regex), col("phone"))
    .otherwise("Unknown")
)

# Find duplicates (excluding "Unknown")
dup = df_silver.filter(col("phone") != "Unknown") \
               .groupBy("phone").count() \
               .filter(col("count") > 1)

display(dup)
print(df_silver.count())
display(df_silver)


In [0]:
#  'email'
# dup=df_silver.groupBy("email").count().filter(col("count")>1)
# display(dup)

from pyspark.sql.functions import col , count , when
count_null = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns])

display(count_null)

df_silver =df_silver.fillna({
    "email" : "UNKNOWN"
})
display(df_silver)
print(df_silver.count())



In [0]:
#address

from pyspark.sql.functions import col , count , when
count_null = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns])

display(count_null)

df_silver =df_silver.fillna({
    "address" : "NOT PROVIDED"
})
display(df_silver)
print(df_silver.count())


In [0]:
# state
dup= df_silver.groupBy("state").count()
display(dup)

In [0]:
#postcode
dup= df_silver.groupBy("postcode").count()
display(dup)

In [0]:
#ahpra_number
from pyspark.sql.functions import col, count , when
check = df_silver.filter(~col("ahpra_number").rlike("^MED"))
display(check)

df_silver = df_silver.withColumn("ahpra_number",when(~col("ahpra_number").rlike("^MED"),None).otherwise(col("ahpra_number")))
display(df_silver)

In [0]:
#salary

from pyspark.sql.functions import abs, col
df_silver = df_silver.withColumn("salary", abs(col("salary")))


display(df_silver)


In [0]:
#  'manager_id

from pyspark.sql.functions import col, count , when
check = df_silver.filter(~col("manager_id").rlike("^EMP"))
display(check)

df_silver = df_silver.withColumn("manager_id",when(~col("manager_id").rlike("^EMP"),None).otherwise(col("manager_id")))
display(df_silver)

In [0]:
#  'created_at
dup= df_silver.groupBy("created_at").count()
display(dup)


In [0]:
#  'status

dup= df_silver.groupBy("status").count()
display(dup)

In [0]:

#  'first_name',
#  'last_name',
#  'role',
#  'facility_id',
#  'employment_type',
#  'start_date',
#  'end_date',
#  'phone',
#  'email',
#  'address',
#  'state',
#  'postcode',
#  'ahpra_number',
#  'salary',
#  'manager_id',
#  'created_at',
#  'status',

silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
            .mode("overwrite")\
.saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
# load to s3
df.write.format("delta")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")

Gold processing

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")

In [0]:
df_gold = df_silver.select(
 'employee_id',
 'first_name',
  'last_name',
  'role',
  'facility_id',
  'employment_type',
  'start_date',
  'end_date',
  'phone',
  'email',
  'address',
  'state',
  'postcode',
  'ahpra_number',
  'salary',
  'manager_id',
  'created_at',
  'status',
)
display(df_gold)

In [0]:
# df_gold.write\
#     .format("delta")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"{catalog}.{gold_schema}.dim_{data_source}")

In [0]:
df_gold.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
df = spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_{data_source};")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_employees")
# Create DataFrame from source table with correct fully-qualified name
df_child_payments = (
    spark.table("regis_healthcare.gold.sb_dim_employees")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_payments.alias("source"),
    condition="target.employee_id = source.employee_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from {catalog}.{gold_schema}.dim_{data_source};")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_{data_source};")
print(sb_dim_df.count())

gold load to s3

In [0]:
# df_gold.write\
#     .format("delta")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .save(f"s3://regis-healthcare/gold-delta-table/dim_{data_source}")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_{data_source}"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = df

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.employee_id = source.employee_id"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)